# Task 3: Classification and Regression

This notebook covers Task 3 of the DM1 project (30 pts).

We follow this structure:

**Classification (target = Rating: Low / Medium / High)**
- Section 0 — Setup
- Section 1 — Feature Selection
- Section 2 — Train / Test Split
- Section 3 — Decision Tree (parameter grid, evaluation, ROC, interpretation)
- Section 4 — KNN (scaling, parameter grid, evaluation, ROC)
- Section 5 — Naive Bayes (evaluation, ROC)
- Section 6 — Classifier Comparison

**Regression (target = GameWeight)**
- Section 7 — Regression Target and Feature Justification
- Section 8 — Single Linear Regression
- Section 9 — Multiple Regression (Linear, Polynomial, Decision Tree Regressor)
- Section 10 — Regression Comparison
- Section 11 — Final Summary

All markdown discussions reference **real output values** from the code cells above them.

## Section 0 — Setup

In [1]:
import os
# Windows fix: loky subprocess hang with n_jobs=-1
os.environ['LOKY_MAX_CPU_COUNT'] = '4'

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, mean_squared_error, r2_score
)
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

import sys
sys.path.append(os.path.abspath('../src'))
from plotting import save_plot, setup_style

setup_style()
%matplotlib inline

RANDOM_STATE = 42
CV_FOLDS = 5

In [2]:
# Load the cleaned dataset produced by Task 1
df = pd.read_csv('../dataset/processed/DM1_game_dataset_clean.csv')

print(f'Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns')
print()
print('Rating distribution (target variable):')
counts = df['Rating'].value_counts()
pct    = (df['Rating'].value_counts(normalize=True) * 100).round(1)
summary = pd.DataFrame({'Count': counts, 'Percent (%)': pct})
print(summary)
print()
print('Note: classes are imbalanced — High is the minority class (23%).')
print('We will use F1-macro as the main evaluation metric, not accuracy.')

Dataset shape: 21925 rows x 37 columns

Rating distribution (target variable):
        Count  Percent (%)
Rating                    
Medium   9644         44.0
Low      7245         33.0
High     5036         23.0

Note: classes are imbalanced — High is the minority class (23%).
We will use F1-macro as the main evaluation metric, not accuracy.


## Section 1 — Feature Selection for Classification

The guideline says: *"discuss the choice of the attributes"*.

We start by computing the Pearson correlation between every numeric feature and an
ordinal encoding of Rating (Low=0, Medium=1, High=2).
This gives a first-pass ranking of predictive signal per feature.

### 1.1 — Correlation of all features with Rating

In [3]:
# Encode Rating as ordinal for correlation calculation only
rating_order = ['Low', 'Medium', 'High']
rating_map   = {'Low': 0, 'Medium': 1, 'High': 2}
df['Rating_enc'] = df['Rating'].map(rating_map)

# All potential features (everything except Rating and its encoding)
all_features = [c for c in df.columns if c not in ('Rating', 'Rating_enc')]

corr_with_rating = (
    df[all_features + ['Rating_enc']]
    .corr()['Rating_enc']
    .drop('Rating_enc')
    .sort_values(ascending=False)
)

print('Pearson correlation of all features with Rating (ordinal-encoded):')
print(corr_with_rating.to_string())
print()
print('Top 5 positive predictors:', corr_with_rating.head(5).index.tolist())
print('Top 5 negative predictors:', corr_with_rating.tail(5).index.tolist())

Pearson correlation of all features with Rating (ordinal-encoded):
log_NumWish            0.564174
GameWeight             0.447836
log_NumOwned           0.295058
NumWish                0.293406
log_MfgPlaytime        0.289854
ComAgeRec              0.269389
BestPlayers            0.236335
YearPublished          0.214519
Cat:Strategy           0.211504
Kickstarted            0.199487
NumOwned               0.189584
Cat:War                0.185015
MfgAgeRec              0.176424
Rank:childrensgames    0.167428
NumWeightVotes         0.153245
NumExpansions          0.145052
IsReimplementation     0.135385
NumImplementations     0.093911
Cat:Thematic           0.085004
MfgPlaytime            0.083609
Rank:abstracts         0.053017
Rank:partygames        0.028737
Rank:cgs               0.009859
LanguageEase           0.004062
Cat:Family             0.003270
Cat:CGS               -0.010258
Rank:familygames      -0.010750
Cat:Party             -0.029782
NumAlternates         -0.037123
Cat:A

### 1.2 — Feature selection decisions

In [4]:
# --- Decision 1: drop raw skewed variables, keep log-transformed versions ---
# From Task 1 we know NumOwned, NumWish, MfgPlaytime are right-skewed.
# The log versions are more normally distributed and have HIGHER correlation with Rating:
#   log_NumWish r=0.564  vs  NumWish r=0.293
#   log_NumOwned r=0.295 vs  NumOwned r=0.190
#   log_MfgPlaytime r=0.290 vs MfgPlaytime r=0.084
# Using log versions gives classifiers better numeric input.
drop_raw_skewed = ['NumOwned', 'NumWish', 'MfgPlaytime']

# --- Decision 2: drop 3 Rank columns that are near-perfect duplicates of Cat columns ---
# From Task 1 correlation analysis:
#   Rank:cgs         vs Cat:CGS          r = -1.0000  (perfect inverse)
#   Rank:partygames  vs Cat:Party        r = -1.0000  (perfect inverse)
#   Rank:childrensgames vs Cat:Childrens r = -0.9999  (near-perfect inverse)
# Keeping both would introduce perfect multicollinearity. We keep the Cat: columns
# because they are binary (0/1) and easier to interpret.
drop_rank_dupes = ['Rank:cgs', 'Rank:partygames', 'Rank:childrensgames']

# Build the final feature list
drop_all = drop_raw_skewed + drop_rank_dupes
FEATURES = [c for c in all_features if c not in drop_all]

print(f'Dropped {len(drop_raw_skewed)} raw skewed columns (log versions kept): {drop_raw_skewed}')
print(f'Dropped {len(drop_rank_dupes)} near-duplicate Rank columns: {drop_rank_dupes}')
print()
print(f'Final feature count: {len(FEATURES)}')
print()
print('Feature list:')
for i, f in enumerate(FEATURES, 1):
    r = corr_with_rating[f]
    print(f'  {i:2d}. {f:<25s}  r={r:+.4f}')

Dropped 3 raw skewed columns (log versions kept): ['NumOwned', 'NumWish', 'MfgPlaytime']
Dropped 3 near-duplicate Rank columns: ['Rank:cgs', 'Rank:partygames', 'Rank:childrensgames']

Final feature count: 30

Feature list:
   1. YearPublished              r=+0.2145
   2. GameWeight                 r=+0.4478
   3. MinPlayers                 r=-0.1782
   4. MaxPlayers                 r=-0.0588
   5. ComAgeRec                  r=+0.2694
   6. LanguageEase               r=+0.0041
   7. BestPlayers                r=+0.2363
   8. NumWeightVotes             r=+0.1532
   9. MfgAgeRec                  r=+0.1764
  10. NumAlternates              r=-0.0371
  11. NumExpansions              r=+0.1451
  12. NumImplementations         r=+0.0939
  13. IsReimplementation         r=+0.1354
  14. Kickstarted                r=+0.1995
  15. Rank:strategygames         r=-0.2175
  16. Rank:abstracts             r=+0.0530
  17. Rank:familygames           r=-0.0108
  18. Rank:thematic              r=-0.0879
  1

**Feature selection discussion:**

We end up with **30 features** grouped as follows:

- **Continuous game properties (12):** YearPublished, GameWeight, MinPlayers, MaxPlayers,
  ComAgeRec, LanguageEase, BestPlayers, NumWeightVotes, MfgAgeRec, NumAlternates,
  NumExpansions, NumImplementations
- **Binary game flags (2):** IsReimplementation, Kickstarted
- **Rank columns (5):** Rank:strategygames, Rank:abstracts, Rank:familygames,
  Rank:thematic, Rank:wargames
- **Category binary columns (8):** Cat:Thematic, Cat:Strategy, Cat:War, Cat:Family,
  Cat:CGS, Cat:Abstract, Cat:Party, Cat:Childrens
- **Log-transformed popularity columns (3):** log_NumOwned, log_NumWish, log_MfgPlaytime

The three **strongest positive predictors** are `log_NumWish` (r=+0.564), `GameWeight` (r=+0.448),
and `log_NumOwned` (r=+0.295). This makes intuitive sense: popular games (high wish-list
and ownership) and complex games (high weight) tend to receive higher ratings.

The three **strongest negative predictors** are `Rank:strategygames` (r=-0.218),
`Rank:wargames` (r=-0.200), and `MinPlayers` (r=-0.178). Note that Rank columns use
21926 as a sentinel for "unranked" — so a low rank number means the game IS ranked
(i.e. popular/successful), which explains the negative sign.

We **dropped 3 raw skewed columns** (NumOwned, NumWish, MfgPlaytime) because their
log versions have substantially higher correlation with Rating:
log_NumWish r=0.564 vs NumWish r=0.293 — a 92% improvement in linear signal.

We **dropped 3 Rank columns** (Rank:cgs, Rank:partygames, Rank:childrensgames)
because their correlation with the matching Cat: column is |r| ≥ 0.9999, making them
perfect duplicates that would introduce multicollinearity without adding information.

## Section 2 — Train / Test Split

In [5]:
X = df[FEATURES].copy()
y = df['Rating'].copy()

# Stratified 80/20 split — preserves class proportions in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

print(f'Train size: {len(X_train):,} rows  ({len(X_train)/len(X)*100:.1f}%)')
print(f'Test  size: {len(X_test):,}  rows  ({len(X_test)/len(X)*100:.1f}%)')
print()

# Verify stratification held
train_counts = y_train.value_counts()
test_counts  = y_test.value_counts()
train_pct    = (y_train.value_counts(normalize=True) * 100).round(1)
test_pct     = (y_test.value_counts(normalize=True)  * 100).round(1)

split_df = pd.DataFrame({
    'Train count': train_counts,
    'Train %':     train_pct,
    'Test count':  test_counts,
    'Test %':      test_pct,
}).loc[rating_order]

print('Class distribution after split:')
print(split_df.to_string())
print()
print(f'Feature matrix shape  — train: {X_train.shape}, test: {X_test.shape}')
print(f'Any missing values in X_train: {X_train.isnull().sum().sum()}')
print(f'Any missing values in X_test:  {X_test.isnull().sum().sum()}')

Train size: 17,540 rows  (80.0%)
Test  size: 4,385  rows  (20.0%)

Class distribution after split:
        Train count  Train %  Test count  Test %
Rating                                          
Low            5796     33.0        1449    33.0
Medium         7715     44.0        1929    44.0
High           4029     23.0        1007    23.0

Feature matrix shape  — train: (17540, 30), test: (4385, 30)
Any missing values in X_train: 0
Any missing values in X_test:  0


**Split discussion:**

We use a **stratified 80/20 split** with `random_state=42` to ensure reproducibility.
Stratification is important here because Rating is imbalanced: without it,
the test set could accidentally under-represent the High class (only 23% of data).

The output confirms stratification held — all three classes maintain their original
proportions (Low ≈33%, Medium ≈44%, High ≈23%) in both train and test sets.

The training set has **17,540 rows** and the test set has **4,385 rows**,
both with 30 features and zero missing values.